In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_squared_error

# Display plots inside Jupyter Notebook
%matplotlib inline

In [ ]:
df = pd.read_excel("Production_Data.xlsx", skiprows=[1])

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(df["Month"],
         df["Oil Rate"],
         marker='o',
         color='blue',
         linewidth=2)

plt.title("Oil Production Rate vs Time", fontsize=16)

plt.xlabel("Time (Months)", fontsize=12)
plt.ylabel("Oil Rate (BOPD)", fontsize=12)

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(df["Month"],
         df["Water Cut"],
         marker='s',
         color='red',
         linewidth=2)

plt.title("Water Cut vs Time", fontsize=16)
plt.xlabel("Time (Months)", fontsize=12)
plt.ylabel("Water Cut (%)", fontsize=12)

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(df["Month"],
         df["GOR"],
         marker='^',
         color='green',
         linewidth=2)

plt.title("GOR vs Time", fontsize=16)
plt.xlabel("Time (Months)", fontsize=12)
plt.ylabel("GOR (v/v)", fontsize=12)

plt.grid(True)

plt.show()

In [ ]:
# Independent variable (time)
t = df["Month"].values

# Dependent variable (oil production rate)
q = df["Oil Rate"].values

print("Time (Months):")
print(t)

print("\nOil Rate (BOPD):")
print(q)

In [ ]:
# Exponential Decline Model
def exponential_decline(t, qi, Di):
    """
    Arps Exponential Decline Model

    Parameters:
    t  : Time (months)
    qi : Initial oil production rate (BOPD)
    Di : Initial decline rate (1/month)

    Returns:
    Predicted oil production rate
    """
    return qi * np.exp(-Di * t)

In [ ]:
# Initial guess for qi and Di
initial_guess = [1402, 0.05]

# Fit exponential decline model
params_exp, covariance_exp = curve_fit(
    exponential_decline,
    t,
    q,
    p0=initial_guess
)

# Extract fitted parameters
qi_exp, Di_exp = params_exp

print("Exponential Decline Parameters")
print("--------------------------------")
print(f"Initial Production Rate (qi) = {qi_exp:.2f} BOPD")
print(f"Initial Decline Rate (Di)    = {Di_exp:.6f} per month")

In [ ]:
q_exp = exponential_decline(t, qi_exp, Di_exp)

In [ ]:
# Display actual and predicted values
comparison = pd.DataFrame({
    "Month": t,
    "Actual_Oil_Rate": q,
    "Predicted_Exponential": q_exp
})

comparison.head(10)

In [ ]:
comparison["Predicted_Exponential"] = comparison["Predicted_Exponential"].round(2)

comparison.head(10)

In [ ]:
plt.figure(figsize=(10,6))

# Actual production data
plt.scatter(t, q,
            color='blue',
            label='Actual Data',
            s=60)

# Exponential fit
plt.plot(t, q_exp,
         color='red',
         linewidth=2,
         label='Exponential Fit')

plt.title("Exponential Decline Curve Fit", fontsize=16)
plt.xlabel("Time (Months)", fontsize=12)
plt.ylabel("Oil Rate (BOPD)", fontsize=12)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Calculate R²
r2_exp = r2_score(q, q_exp)

# Calculate RMSE
rmse_exp = np.sqrt(mean_squared_error(q, q_exp))

print("Exponential Model Performance")
print("-----------------------------")
print(f"R²   = {r2_exp:.4f}")
print(f"RMSE = {rmse_exp:.2f} BOPD")

In [ ]:
# Hyperbolic Decline Model
def hyperbolic_decline(t, qi, Di, b):
    """
    Arps Hyperbolic Decline Model

    Parameters:
    t  : Time (months)
    qi : Initial oil production rate (BOPD)
    Di : Initial decline rate (1/month)
    b  : Hyperbolic decline exponent

    Returns:
    Predicted oil production rate
    """
    return qi / ((1 + b * Di * t) ** (1 / b))

In [ ]:
# Initial guess
initial_guess = [1402, 0.05, 0.5]

# Fit Hyperbolic Decline Model
params_hyp, covariance_hyp = curve_fit(
    hyperbolic_decline,
    t,
    q,
    p0=initial_guess,
    bounds=([0, 0, 0], [np.inf, np.inf, 2])
)

# Extract fitted parameters
qi_hyp, Di_hyp, b_hyp = params_hyp

print("Hyperbolic Decline Parameters")
print("--------------------------------")
print(f"Initial Production Rate (qi) = {qi_hyp:.2f} BOPD")
print(f"Initial Decline Rate (Di)    = {Di_hyp:.6f} per month")
print(f"Decline Exponent (b)         = {b_hyp:.4f}")

In [ ]:
# Predicted oil rates using Hyperbolic model
q_hyp = hyperbolic_decline(t, qi_hyp, Di_hyp, b_hyp)

In [ ]:
comparison["Predicted_Hyperbolic"] = q_hyp.round(2)

comparison.head(10)

In [ ]:
plt.figure(figsize=(10,6))

# Actual production data
plt.scatter(t, q,
            color='blue',
            s=60,
            label='Actual Data')

# Hyperbolic fit
plt.plot(t, q_hyp,
         color='green',
         linewidth=2,
         label='Hyperbolic Fit')

plt.title("Hyperbolic Decline Curve Fit", fontsize=16)
plt.xlabel("Time (Months)", fontsize=12)
plt.ylabel("Oil Rate (BOPD)", fontsize=12)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Calculate R²
r2_hyp = r2_score(q, q_hyp)

# Calculate RMSE
rmse_hyp = np.sqrt(mean_squared_error(q, q_hyp))

print("Hyperbolic Model Performance")
print("-----------------------------")
print(f"R²   = {r2_hyp:.4f}")
print(f"RMSE = {rmse_hyp:.2f} BOPD")

In [ ]:
# Harmonic Decline Model
def harmonic_decline(t, qi, Di):
    """
    Arps Harmonic Decline Model

    Parameters:
    t  : Time (months)
    qi : Initial oil production rate (BOPD)
    Di : Initial decline rate (1/month)

    Returns:
    Predicted oil production rate
    """
    return qi / (1 + Di * t)

In [ ]:
# Initial guess
initial_guess = [1402, 0.05]

# Fit Harmonic Decline Model
params_har, covariance_har = curve_fit(
    harmonic_decline,
    t,
    q,
    p0=initial_guess
)

# Extract parameters
qi_har, Di_har = params_har

print("Harmonic Decline Parameters")
print("---------------------------")
print(f"Initial Production Rate (qi) = {qi_har:.2f} BOPD")
print(f"Initial Decline Rate (Di)    = {Di_har:.6f} per month")

In [ ]:
# Predicted oil rates using Harmonic model
q_har = harmonic_decline(t, qi_har, Di_har)

In [ ]:
comparison["Predicted_Harmonic"] = q_har.round(2)

comparison.head(10)

In [ ]:
plt.figure(figsize=(10,6))

# Actual production
plt.scatter(t, q,
            color='blue',
            s=60,
            label='Actual Data')

# Harmonic fit
plt.plot(t, q_har,
         color='purple',
         linewidth=2,
         label='Harmonic Fit')

plt.title("Harmonic Decline Curve Fit", fontsize=16)
plt.xlabel("Time (Months)", fontsize=12)
plt.ylabel("Oil Rate (BOPD)", fontsize=12)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Calculate R²
r2_har = r2_score(q, q_har)

# Calculate RMSE
rmse_har = np.sqrt(mean_squared_error(q, q_har))

print("Harmonic Model Performance")
print("--------------------------")
print(f"R²   = {r2_har:.4f}")
print(f"RMSE = {rmse_har:.2f} BOPD")

In [ ]:
# Final Model Comparison Table

results = pd.DataFrame({
    "Model": ["Exponential", "Hyperbolic", "Harmonic"],
    "qi (BOPD)": [qi_exp, qi_hyp, qi_har],
    "Di (/month)": [Di_exp, Di_hyp, Di_har],
    "b": [0, b_hyp, 1],
    "R²": [r2_exp, r2_hyp, r2_har],
    "RMSE (BOPD)": [rmse_exp, rmse_hyp, rmse_har]
})

# Round values
results = results.round({
    "qi (BOPD)": 2,
    "Di (/month)": 6,
    "b": 4,
    "R²": 4,
    "RMSE (BOPD)": 2
})

print("Final Model Comparison")

display(
    results.style
    .highlight_max(subset=["R²"], color="lightgreen")
    .highlight_min(subset=["RMSE (BOPD)"], color="lightgreen")
)

In [ ]:
plt.figure(figsize=(12,7))

# Actual production
plt.scatter(t, q,
            color='black',
            s=70,
            label='Actual Data')

# Exponential
plt.plot(t, q_exp,
         color='red',
         linewidth=2,
         label='Exponential')

# Hyperbolic
plt.plot(t, q_hyp,
         color='green',
         linewidth=2,
         label='Hyperbolic')

# Harmonic
plt.plot(t, q_har,
         color='purple',
         linewidth=2,
         label='Harmonic')

plt.title("Comparison of Arps Decline Models", fontsize=16)
plt.xlabel("Time (Months)", fontsize=12)
plt.ylabel("Oil Rate (BOPD)", fontsize=12)

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
# Economic limit (BOPD)
economic_limit = 40

# Start forecasting after the last historical month
forecast_month = int(t[-1]) + 1

forecast_time = []
forecast_rate = []

while True:

    # Hyperbolic forecast
    q_future = hyperbolic_decline(
        forecast_month,
        qi_hyp,
        Di_hyp,
        b_hyp
    )

    # Stop when economic limit is reached
    if q_future <= economic_limit:
        break

    forecast_time.append(forecast_month)
    forecast_rate.append(q_future)

    forecast_month += 1

In [ ]:
forecast_df = pd.DataFrame({
    "Month": forecast_time,
    "Forecast_Oil_Rate (BOPD)": np.round(forecast_rate, 2)
})

forecast_df.head(10)

In [ ]:
forecast_df.tail()

In [ ]:
# First month below the economic limit

final_month = forecast_month
final_rate = hyperbolic_decline(final_month, qi_hyp, Di_hyp, b_hyp)

print("Economic Limit Reached")
print("----------------------")
print(f"Final Forecast Month : {final_month}")
print(f"Oil Rate             : {final_rate:.2f} BOPD")

In [ ]:
plt.figure(figsize=(12,6))

# Historical production
plt.scatter(
    t,
    q,
    color="blue",
    s=60,
    label="Historical Data"
)

# Hyperbolic fit (history)
plt.plot(
    t,
    q_hyp,
    color="green",
    linewidth=2,
    label="Hyperbolic Fit"
)

# Forecast
plt.plot(
    forecast_time,
    forecast_rate,
    color="red",
    linewidth=2,
    linestyle="--",
    label="Forecast"
)

# Economic limit
plt.axhline(
    y=40,
    color="black",
    linestyle=":",
    linewidth=2,
    label="Economic Limit (40 BOPD)"
)

plt.title("Historical Production and Forecast using Hyperbolic Decline", fontsize=16)
plt.xlabel("Time (Months)", fontsize=12)
plt.ylabel("Oil Rate (BOPD)", fontsize=12)

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
# Average days in a month
days_per_month = 30.44

# Monthly oil production (barrels)
df["Monthly_Oil_Production (STB)"] = df["Oil Rate"] * days_per_month

df.head()

In [ ]:
# Cumulative oil production
df["Cumulative_Oil_Production (STB)"] = df["Monthly_Oil_Production (STB)"].cumsum()

df.head(10)

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(
    df["Month"],
    df["Cumulative_Oil_Production (STB)"],
    color="blue",
    linewidth=2
)

plt.title("Historical Cumulative Oil Production", fontsize=16)
plt.xlabel("Time (Months)")
plt.ylabel("Cumulative Production (STB)")

plt.grid(True)

plt.show()

In [ ]:
# Monthly forecast oil production (STB)

forecast_df["Monthly_Oil_Production (STB)"] = (
    forecast_df["Forecast_Oil_Rate (BOPD)"] * days_per_month
)

forecast_df.head()

In [ ]:
# Last historical cumulative production
last_cum = df["Cumulative_Oil_Production (STB)"].iloc[-1]

# Forecast cumulative production
forecast_df["Cumulative_Oil_Production (STB)"] = (
    forecast_df["Monthly_Oil_Production (STB)"].cumsum() + last_cum
)

forecast_df.head()

In [ ]:
forecast_df.tail()

In [ ]:
ultimate_recovery = forecast_df["Cumulative_Oil_Production (STB)"].iloc[-1]

print("Ultimate Recovery Summary")
print("-------------------------")
print(f"Historical Production Period : {int(df['Month'].iloc[-1])} months")
print(f"Forecast Production Period   : {int(forecast_df['Month'].iloc[-1])} months")
print(f"Economic Limit               : 40 BOPD")
print(f"Estimated Ultimate Recovery  : {ultimate_recovery:,.2f} STB")

In [ ]:
plt.figure(figsize=(12,6))

# Historical cumulative
plt.plot(
    df["Month"],
    df["Cumulative_Oil_Production (STB)"],
    color="blue",
    linewidth=2,
    label="Historical"
)

# Forecast cumulative
plt.plot(
    forecast_df["Month"],
    forecast_df["Cumulative_Oil_Production (STB)"],
    color="red",
    linestyle="--",
    linewidth=2,
    label="Forecast"
)

plt.title("Historical and Forecast Cumulative Oil Production", fontsize=16)
plt.xlabel("Time (Months)")
plt.ylabel("Cumulative Oil Production (STB)")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
# Historical data
history = pd.DataFrame({
    "Month": df["Month"],
    "Oil_Rate": df["Oil Rate"]
})

# Forecast data
future = pd.DataFrame({
    "Month": forecast_df["Month"],
    "Oil_Rate": forecast_df["Forecast_Oil_Rate (BOPD)"]
})

# Combine both
combined = pd.concat([history, future], ignore_index=True)

combined.head()

In [ ]:
combined["Year"] = ((combined["Month"] - 1) // 12) + 1

combined.head(15)

In [ ]:
combined["Monthly_Production (STB)"] = (
    combined["Oil_Rate"] * days_per_month
)

In [ ]:
yearly_summary = combined.groupby("Year").agg(
    Average_Oil_Rate=("Oil_Rate", "mean"),
    Yearly_Production=("Monthly_Production (STB)", "sum")
)

In [ ]:
yearly_summary["Yearly_Cumulative_Production"] = (
    yearly_summary["Yearly_Production"].cumsum()
)

In [ ]:
yearly_summary = yearly_summary.round({
    "Average_Oil_Rate": 2,
    "Yearly_Production": 2,
    "Yearly_Cumulative_Production": 2
})

print("Yearly Production Summary")

display(yearly_summary)

In [ ]:
# Figure
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle("Yearly Production Analysis", fontsize=22, fontweight="bold")

# -----------------------------
# 1. Yearly Average Production Rate
# -----------------------------
axes[0].bar(
    yearly_summary.index,
    yearly_summary["Average_Oil_Rate"],
    color="royalblue",
    width=0.6,
    label="Yearly Average Rate (BOPD)"
)

axes[0].set_title("Yearly Average Production Rate", fontsize=16)
axes[0].set_ylabel("Average Rate (BOPD)")
axes[0].grid(axis='y', linestyle='--', alpha=0.4)
axes[0].legend()

for i, value in enumerate(yearly_summary["Average_Oil_Rate"]):
    axes[0].text(
        yearly_summary.index[i],
        value + 10,
        f"{value:.0f}",
        ha='center',
        color='navy',
        fontsize=10,
        fontweight='bold'
    )

# -----------------------------
# 2. Yearly Oil Production
# -----------------------------
axes[1].bar(
    yearly_summary.index,
    yearly_summary["Yearly_Production"],
    color="green",
    width=0.6,
    label="Yearly Production (STB)"
)

axes[1].set_title("Yearly Oil Production", fontsize=16)
axes[1].set_ylabel("Production (STB)")
axes[1].grid(axis='y', linestyle='--', alpha=0.4)
axes[1].legend()

for i, value in enumerate(yearly_summary["Yearly_Production"]):
    axes[1].text(
        yearly_summary.index[i],
        value + 5000,
        f"{value:,.0f}",
        ha='center',
        color='darkgreen',
        fontsize=10,
        fontweight='bold'
    )

# -----------------------------
# 3. Yearly Cumulative Production
# -----------------------------
axes[2].bar(
    yearly_summary.index,
    yearly_summary["Yearly_Cumulative_Production"],
    color="red",
    width=0.6,
    label="Cumulative Production (STB)"
)

axes[2].set_title("Yearly Cumulative Oil Production", fontsize=16)
axes[2].set_ylabel("Cumulative Production (STB)")
axes[2].set_xlabel("Year")
axes[2].grid(axis='y', linestyle='--', alpha=0.4)
axes[2].legend()

for i, value in enumerate(yearly_summary["Yearly_Cumulative_Production"]):
    axes[2].text(
        yearly_summary.index[i],
        value + 15000,
        f"{value:,.0f}",
        ha='center',
        color='firebrick',
        fontsize=10,
        fontweight='bold'
    )

plt.tight_layout(rect=[0, 0, 1, 0.97])

plt.show()
